# Survey Populations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import utilities.plot_settings

from matplotlib import rc

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

## Load observed data

Load the ATNF Catalogue.

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../examples/data/atnf_full_nobinary_25-04-2023.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

In [ ]:
df_atnf = df_atnf.drop(
    columns=[
        "#",
        "PMRA",
        "PMDEC",
        "PX",
        "POSEPOCH",
        "RAJD",
        "DECJD",
        "DM",
        "W50",
        "W10",
        "TAU_SC",
        "S400",
        "S2000",
        "DIST",
        "DIST_DM",
        "ZZ",
        "XX",
        "YY",
        "PSR",
        "Unnamed: 27_level_0",
    ],
    level=0,
)

len(df_atnf)

In [ ]:
# Select only those with measured period values.
df_atnf = df_atnf[~df_atnf["P0"]["[s]"].isin(["NAN"])]

# Remove those objects that are in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]
df_atnf = df_atnf[
    ~df_atnf["ASSOC"]["Unnamed: 24_level_1"].str.match("|".join(discard))
]

len(df_atnf)

In [ ]:
# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"]["[s]"].to_numpy().astype(np.float64) > 0.01]

len(df_atnf)

In [ ]:
df_atnf = df_atnf[
    (df_atnf["P1"]["[s/s]"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"]["[s/s]"].isin(["NAN"]))
]

In [ ]:
# Parks multibeam pulsar survey database.
df_atnf_pk = df_atnf[
    df_atnf["SURVEY"]["Unnamed: 25_level_1"].str.contains("pksmb")
]

l_pk_obs = df_atnf_pk["Gl"]["[deg]"].to_numpy().astype(np.float64)
b_pk_obs = df_atnf_pk["Gb"]["[deg]"].to_numpy().astype(np.float64)
P_pk_obs = df_atnf_pk["P0"]["[s]"].to_numpy().astype(np.float64)
Pdot_pk_obs = df_atnf_pk["P1"]["[s/s]"].to_numpy().astype(np.float64)
S1400_pk_obs = df_atnf_pk["S1400"]["[mJy]"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_pk_obs[(l_pk_obs > 180.0) & (l_pk_obs < 360.0)] = (
    l_pk_obs[(l_pk_obs > 180.0) & (l_pk_obs < 360.0)] - 360.0
)

# Select only pulsars falling in the Parkes multibeam sky coverage where completness is above 90%.
cond = (l_pk_obs > -100.0) & (l_pk_obs < 50.0) & (np.abs(b_pk_obs) < 5.0)

l_pk_obs = l_pk_obs[cond]
b_pk_obs = b_pk_obs[cond]
P_pk_obs = P_pk_obs[cond]
Pdot_pk_obs = Pdot_pk_obs[cond]
S1400_pk_obs = S1400_pk_obs[cond]

number_pk = len(l_pk_obs)

print(number_pk)

In [ ]:
# Swinburne multibeam pulsar survey database.
df_atnf_sw = df_atnf[
    df_atnf["SURVEY"]["Unnamed: 25_level_1"].str.contains("pkssw")
]

l_sw_obs = df_atnf_sw["Gl"]["[deg]"].to_numpy().astype(np.float64)
b_sw_obs = df_atnf_sw["Gb"]["[deg]"].to_numpy().astype(np.float64)
P_sw_obs = df_atnf_sw["P0"]["[s]"].to_numpy().astype(np.float64)
Pdot_sw_obs = df_atnf_sw["P1"]["[s/s]"].to_numpy().astype(np.float64)
S1400_sw_obs = df_atnf_sw["S1400"]["[mJy]"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_sw_obs[(l_sw_obs > 180.0) & (l_sw_obs < 360.0)] = (
    l_sw_obs[(l_sw_obs > 180.0) & (l_sw_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the Swinburne sky coverage where completness is above 90%.
cond = (l_sw_obs > -100.0) & (l_sw_obs < 50.0)

l_sw_obs = l_sw_obs[cond]
b_sw_obs = b_sw_obs[cond]
P_sw_obs = P_sw_obs[cond]
Pdot_sw_obs = Pdot_sw_obs[cond]
S1400_sw_obs = S1400_sw_obs[cond]

l_all_obs = np.concatenate((l_pk_obs, l_sw_obs))
b_all_obs = np.concatenate((b_pk_obs, b_sw_obs))
P_all_obs = np.concatenate((P_pk_obs, P_sw_obs))
Pdot_all_obs = np.concatenate((Pdot_pk_obs, Pdot_sw_obs))
S1400_all_obs = np.concatenate((S1400_pk_obs, S1400_sw_obs))

number_sw = len(l_sw_obs)
print(number_sw)

In [ ]:
# HTRU multibeam pulsar survey database.
df_atnf_htru = df_atnf[
    df_atnf["SURVEY"]["Unnamed: 25_level_1"].str.contains("htru_pks")
]

l_htru_obs = df_atnf_htru["Gl"]["[deg]"].to_numpy().astype(np.float64)
b_htru_obs = df_atnf_htru["Gb"]["[deg]"].to_numpy().astype(np.float64)
P_htru_obs = df_atnf_htru["P0"]["[s]"].to_numpy().astype(np.float64)
Pdot_htru_obs = df_atnf_htru["P1"]["[s/s]"].to_numpy().astype(np.float64)
S1400_htru_obs = df_atnf_htru["S1400"]["[mJy]"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] = (
    l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the Swinburne sky coverage where completness is above 90%.
cond = (
    (l_htru_obs > -120.0) & (l_htru_obs < 30.0) & (np.abs(b_htru_obs) < 15.0)
)

l_htru_obs = l_htru_obs[cond]
b_htru_obs = b_htru_obs[cond]
P_htru_obs = P_htru_obs[cond]
Pdot_htru_obs = Pdot_htru_obs[cond]
S1400_htru_obs = S1400_htru_obs[cond]

l_all_obs = np.concatenate((l_pk_obs, l_sw_obs, l_htru_obs))
b_all_obs = np.concatenate((b_pk_obs, b_sw_obs, b_htru_obs))
P_all_obs = np.concatenate((P_pk_obs, P_sw_obs, P_htru_obs))
Pdot_all_obs = np.concatenate((Pdot_pk_obs, Pdot_sw_obs, Pdot_htru_obs))
S1400_all_obs = np.concatenate((S1400_pk_obs, S1400_sw_obs, S1400_htru_obs))

number_htru = len(l_htru_obs)
print(number_htru)

In [ ]:
print(f"Number of pulsars detected by Parks multibeam: {number_pk}")
print(f"Number of pulsars detected by Swinburne: {number_sw}")
print(f"Number of pulsars detected by HTRU: {number_htru}")

In [ ]:
print(
    f"Number of pulsars detected by Parks multibeam without Pdot measurement: {np.isnan(Pdot_pk_obs).sum()}"
)
print(
    f"Number of pulsars detected by Swinburne Parkes multibeam without Pdot measurement: {np.isnan(Pdot_sw_obs).sum()}"
)
print(
    f"Number of pulsars detected by HTRU without Pdot measurement: {np.isnan(Pdot_htru_obs).sum()}"
)

In [ ]:
print(
    f"Number of pulsars detected by Parks multibeam without S1400 measurement: {np.isnan(S1400_pk_obs).sum()}"
)
print(
    f"Number of pulsars detected by Swinburne Parkes multibeam without S1400 measurement: {np.isnan(S1400_sw_obs).sum()}"
)
print(
    f"Number of pulsars detected by HTRU without S1400 measurement: {np.isnan(S1400_htru_obs).sum()}"
)

## Plotting the observations

Sky positions.

In [ ]:
colors = ["#fde725", "dodgerblue", "#440154"]

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

plt.plot(
    l_pk_obs,
    b_pk_obs,
    linestyle="None",
    marker="o",
    color=colors[0],
    markersize=8,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    l_sw_obs,
    b_sw_obs,
    linestyle="None",
    marker="o",
    color=colors[1],
    markersize=8,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    l_htru_obs,
    b_htru_obs,
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color=colors[2],
    markersize=8,
    alpha=0.3,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.set_xlim(-130.0, 80.0)
ax.set_ylim(-50.0, 50.0)
ax.set_xlabel(r"Galactic longitude $l$ [deg]")
ax.set_ylabel(r"Galactic latitude $b$ [deg]")
ax.legend(frameon=True, loc="best")
ax.grid()

plt.tight_layout()
plt.savefig(
    "../paper_plots/plots/observed_coord.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

Edot lines.

In [ ]:
I_NS = 1.36e45
R_NS = 1.1e6
c = 2.998e10

Pdot_Edot_lines = np.zeros((6, 51))

Edot_log = np.linspace(28, 38, 6)
print(Edot_log)

In [ ]:
P_log = np.linspace(-3, 2, 51)

In [ ]:
for i in range(len(Edot_log)):
    Pdot_Edot_lines[i] = (
        10 ** Edot_log[i] * (10**P_log) ** 3 / (4 * np.pi**2 * I_NS)
    )

B lines.

In [ ]:
Pdot_B_lines = np.zeros((5, 51))

B_log = np.linspace(10, 14, 5)
print(B_log)

In [ ]:
for i in range(len(B_log)):
    Pdot_B_lines[i] = (
        np.pi**2
        * (10 ** B_log[i]) ** 2
        * (R_NS**6)
        / (I_NS * 10**P_log * c**3)
    )

PPdot diagrams.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))

for i in range(len(Edot_log)):
    ax.plot(
        10**P_log,
        Pdot_Edot_lines[i],
        linestyle="-",
        # fillstyle="none",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
for i in range(len(B_log)):
    ax.plot(
        10**P_log,
        Pdot_B_lines[i],
        linestyle="-",
        # fillstyle="none",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
ax.plot(
    P_pk_obs,
    Pdot_pk_obs,
    linestyle="None",
    marker="o",
    color=colors[0],
    markersize=9,
    alpha=1.0,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    P_sw_obs,
    Pdot_sw_obs,
    linestyle="None",
    marker="o",
    color=colors[1],
    markersize=9,
    alpha=1.0,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    P_htru_obs,
    Pdot_htru_obs,
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color=colors[2],
    markersize=9,
    alpha=0.3,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-3, 100.0)
ax.set_ylim(1.0e-21, 1.0e-9)

ax.text(
    0.41,
    0.015,
    r"$10^{28} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.275,
    0.015,
    r"$10^{30} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.143,
    0.015,
    r"$10^{32} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.015,
    r"$10^{34} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.181,
    r"$10^{36} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.347,
    r"$10^{38} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

ax.text(
    0.89,
    0.005,
    r"$10^{10} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.172,
    r"$10^{11} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.340,
    r"$10^{12} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.506,
    r"$10^{13} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.672,
    r"$10^{14} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

plt.xlabel(r"Period $P$ [s]")
plt.ylabel(r"Period derivative $\dot{P}$ [s$\,{\rm s}^{-1}$]")
ax.legend(frameon=True, loc=2)
# ax.grid()

plt.tight_layout()
plt.savefig(
    "../paper_plots/plots/observed_ppdot.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

Plotting the observed fluxes at 1.4 GHz.

In [ ]:
print(min(np.log(S1400_pk_obs)), np.log(max(S1400_pk_obs)))

In [ ]:
bin_edges = np.arange(-4, 7, 0.5)
print(bin_edges)

In [ ]:
# New X-axis values.
x = np.linspace(-4, 7, 1000)

# Remove NaNs and estimate the PDF using a Gaussian kernel.
S1400_pk_obs = S1400_pk_obs[~np.isnan(S1400_pk_obs)]
kde_pk_obs = stats.gaussian_kde(np.log(S1400_pk_obs))

S1400_sw_obs = S1400_sw_obs[~np.isnan(S1400_sw_obs)]
kde_sw_obs = stats.gaussian_kde(np.log(S1400_sw_obs))

S1400_htru_obs = S1400_htru_obs[~np.isnan(S1400_htru_obs)]
kde_htru_obs = stats.gaussian_kde(np.log(S1400_htru_obs))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

# Histograms.
ax.hist(
    np.log(S1400_pk_obs),
    bin_edges,
    align="left",
    density=True,
    color=colors[0],
    alpha=0.5,
    histtype="step",
    lw=4,
    label=r"Observed PMPS",
)
ax.hist(
    np.log(S1400_sw_obs),
    bin_edges,
    align="left",
    density=True,
    color=colors[1],
    alpha=0.5,
    histtype="step",
    lw=4,
    label=r"Observed SMPS",
)
ax.hist(
    np.log(S1400_htru_obs),
    bin_edges,
    align="left",
    density=True,
    color=colors[2],
    alpha=0.5,
    histtype="step",
    lw=4,
    label=r"Observed HTRU",
)

# KDE density plots.
ax.plot(
    x,
    kde_pk_obs(x),
    color=colors[0],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE observed PMPS",
)
ax.plot(
    x,
    kde_sw_obs(x),
    color=colors[1],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE observed SMPS",
)
ax.plot(
    x,
    kde_htru_obs(x),
    color=colors[2],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE observed HTRU",
)

ax.set_xlim(-4, 7)
ax.set_ylim(0.0, 0.5)
ax.set_xlabel(r"Mean flux density log $S_{{\rm mean}, 1400}$ [mJy]")
ax.set_ylabel(r"Normalised pulsar count")
ax.legend(frameon=True, loc="best")

plt.tight_layout()
plt.savefig(
    "../paper_plots/plots/flux_hist_observed.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()